In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12



In [9]:
df = pd.read_csv("city_day.csv")
print("Shape:", df.shape)
df.head()

Shape: (29531, 16)


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


In [10]:
print("="*50)
print("DATASET INFORMATION")
print("="*50)
print(f"Total Rows    : {df.shape[0]}")
print(f"Total Columns : {df.shape[1]}")
print(f"\nColumn Names  : {list(df.columns)}")
print(f"\nDate Range    : {df['Date'].min()} to {df['Date'].max()}")
print(f"\nCities Covered: {df['City'].nunique()} cities")
print(f"\nCity List     : {sorted(df['City'].unique())}")

DATASET INFORMATION
Total Rows    : 29531
Total Columns : 16

Column Names  : ['City', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket']

Date Range    : 2015-01-01 to 2020-07-01

Cities Covered: 26 cities

City List     : ['Ahmedabad', 'Aizawl', 'Amaravati', 'Amritsar', 'Bengaluru', 'Bhopal', 'Brajrajnagar', 'Chandigarh', 'Chennai', 'Coimbatore', 'Delhi', 'Ernakulam', 'Gurugram', 'Guwahati', 'Hyderabad', 'Jaipur', 'Jorapokhar', 'Kochi', 'Kolkata', 'Lucknow', 'Mumbai', 'Patna', 'Shillong', 'Talcher', 'Thiruvananthapuram', 'Visakhapatnam']


In [11]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)
print(missing_df[missing_df['Missing Count'] > 0])


            Missing Count  Missing %
Xylene              18109      61.32
PM10                11140      37.72
NH3                 10328      34.97
Toluene              8041      27.23
Benzene              5623      19.04
AQI                  4681      15.85
AQI_Bucket           4681      15.85
PM2.5                4598      15.57
NOx                  4185      14.17
O3                   4022      13.62
SO2                  3854      13.05
NO2                  3585      12.14
NO                   3582      12.13
CO                   2059       6.97


In [ ]:
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())

In [12]:
df.drop(columns=['Xylene'], inplace=True)
df['Date'] = pd.to_datetime(df['Date'])
df['Year']       = df['Date'].dt.year
df['Month']      = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')   # Jan, Feb ...
df['Season']     = df['Month'].map({
    12:'Winter', 1:'Winter', 2:'Winter',
     3:'Spring', 4:'Spring', 5:'Spring',
     6:'Summer', 7:'Summer', 8:'Summer',
     9:'Autumn',10:'Autumn',11:'Autumn'
})

In [13]:
fill_cols = ['PM2.5','PM10','NO','NO2','NOx','NH3',
             'CO','SO2','O3','Benzene','Toluene']

for col in fill_cols:
    df[col] = df.groupby('City')[col].transform(
        lambda x: x.fillna(x.median())
    )

In [14]:
before = len(df)
df.dropna(subset=['AQI', 'AQI_Bucket'], inplace=True)
after  = len(df)
print("Before :", before)
print("After: ",after)

Before : 29531
After:  24850


In [15]:
df.reset_index(drop=True, inplace=True)

In [16]:
print("\n" + "=" * 45)
print("       CLEANING COMPLETE — SUMMARY")
print("=" * 45)
print(f"  Final Shape      : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  Remaining NaN    : {df.isnull().sum().sum()}")
print(f"  Date Range       : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"  Seasons Added    : {sorted(df['Season'].unique())}")
print(f"  New Columns      : Year, Month, Month_Name, Season")


       CLEANING COMPLETE — SUMMARY
  Final Shape      : 24850 rows × 19 columns
  Remaining NaN    : 9719
  Date Range       : 2015-01-01 → 2020-07-01
  Seasons Added    : ['Autumn', 'Spring', 'Summer', 'Winter']
  New Columns      : Year, Month, Month_Name, Season
